# Lesson 07: API and threads

**Goal:** call the pipeline through the HTTP API and keep a conversation in a thread.

**Before you start**
- Theory: [lesson 07](../../docs/lessons/07_api_and_threads.md)
- Python concepts: FastAPI `TestClient`, JSON bodies, `async` endpoints behind a sync client
- Files used: `fixtures/tiny.pdf`, fakes, Qdrant in-memory. Offline, no API key, no cost.

Each step: read the short explanation, fill in the `# TODO` cell, run the check cell below it.

## Setup (just run it)

In [ ]:
import warnings
from pathlib import Path

from fastapi.testclient import TestClient

from docrag.answer.service import AnswerService
from docrag.api.app import create_app
from docrag.embed.base import EmbeddingService
from docrag.embed.fake import FakeEmbedder
from docrag.ingest.pipeline import ingest_pdf
from docrag.llm.fake import FakeProvider
from docrag.llm.proxy import LLMProxy
from docrag.rerank.reranker import FakeReranker
from docrag.shared.config import ExperimentConfig
from docrag.store.qdrant import QdrantStore, make_client
from docrag.store.retriever import Retriever

ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()
)
warnings.filterwarnings(
    "ignore", message="Payload indexes have no effect"
)  # in-memory Qdrant
chunks = ingest_pdf(ROOT / "fixtures" / "tiny.pdf", "tiny")
embeddings = EmbeddingService(FakeEmbedder(dim=64))
store = QdrantStore(make_client(":memory:"), "lesson07", embeddings.dim)
store.ensure_collection()
store.upsert(chunks, embeddings.embed([c.embed_text for c in chunks]))
config = ExperimentConfig(
    name="t", embedder="fake:64", reranker="fake:x", rerank_threshold=0.3
)
service = AnswerService(
    Retriever(embeddings, store, config, FakeReranker()),
    LLMProxy({"fake": FakeProvider()}),
    "fake:m",
)
# TestClient calls the app in-process: no server, no port (like supertest / MockMvc).
client = TestClient(create_app(service, pdf_dir=ROOT / "fixtures"))

## Step 1: POST /query

The body is JSON matching `QueryRequest` (`question`, optional `thread_id`, optional `doc_ids`). FastAPI validates it from the type hints and answers 422 if it's wrong. Without a `thread_id`, a new thread is created and returned.

In [ ]:
q1 = "What was the revenue in Europe in 2024?"
# TODO: r1 = client.post("/query", json={...}) with the question and doc_ids ["tiny"]
r1 = None

In [ ]:
assert r1 is not None, (
    "Hint: client.post('/query', json={'question': q1, 'doc_ids': ['tiny']})"
)
assert r1.status_code == 200, r1.text
body = r1.json()
assert body["status"] == "supported" and body["sources"], (
    "Only verified sources are returned"
)
assert body["sources"][0]["page_image_url"].startswith("/pages/tiny/")
print({k: body[k] for k in ("thread_id", "status", "answer")})
print("OK")

## Step 2: A follow-up in the same thread

Send a second question with the `thread_id` from step 1, then read the thread back with `GET /threads/{thread_id}`. The history goes into the next prompt between the system message and the new question.

In [ ]:
thread_id = body["thread_id"]
# TODO: post "And in 2023?" with this thread_id (and doc_ids ["tiny"]),
#       then history = client.get(f"/threads/{thread_id}").json()
history = []

In [ ]:
roles = [m["role"] for m in history]
assert roles == ["user", "assistant", "user", "assistant"], (
    f"Expected two turns, got {roles}"
)
assert client.get("/threads/does-not-exist").status_code == 404
for m in history:
    print(m["role"].ljust(9), m["content"][:70])
print("OK")

## Observation question

What happens if two requests write to the same thread at the same time?

<details>
<summary>Answer</summary>

`InMemoryThreadStore` guards its dict with a `threading.Lock`, and each request appends its user + assistant messages in **one** `append()` call. So the list never gets corrupted and a pair is never split. But each request read the history *before* the other one finished, so neither answer knows about the other turn, and the order of the two pairs depends on which finished first. A real fix would be optimistic concurrency (a version number per thread, reject stale writes) or serializing requests per thread. With Redis behind the same `ThreadStore` interface the lock would have to move into Redis too.
</details>

## The real code

- Routes: [`src/docrag/api/app.py`](../../src/docrag/api/app.py)
- Thread store: [`src/docrag/threads/store.py`](../../src/docrag/threads/store.py)
- Composition root: [`src/docrag/wiring.py`](../../src/docrag/wiring.py)
- Lesson: [`docs/lessons/07_api_and_threads.md`](../../docs/lessons/07_api_and_threads.md)